# Scientific Validity of ASI for Mean Estimation

This notebook provides empirical evidence that GLIDE's **Active Statistical Inference (ASI)** implementation is statistically valid.

**Setup:** We estimate the mean of a binary outcome (e.g., the hallucination rate of an AI system). We have:
- A pool of `N_TOTAL` samples, each with a proxy label $\tilde{Y}$ (`y_proxy`) and an oracle proxy uncertainty that quantifies how unreliable the proxy is for each individual sample
- A labeling budget of `N_LABELED` samples: we can reveal the true label $Y$ (`y_true_oracle`) for only a fraction of the pool

ASI selects which samples to label using **sampling probabilities** ($\pi_i \propto \text{uncertainty}_i$): samples where the proxy is least reliable are labeled with higher probability. It then corrects for this non-uniform selection via **Inverse Probability Weighting (IPW)**, yielding confidence intervals that are:
1. **Valid** : they cover the true mean at the specified rate regardless of the sampling rule
2. **Shorter** : active sampling concentrates the labeling budget on uncertain samples, producing shorter intervals when the proxy is sufficiently informative

We test these two claims empirically across a range of proxy/true correlation levels.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from glide.confidence_intervals import CLTConfidenceInterval
from glide.core.simulated_datasets import generate_binary_dataset_with_oracle_sampling
from glide.estimators import ASIMeanEstimator, ClassicalMeanEstimator, IPWClassicalMeanEstimator
from glide.samplers.active import ActiveSampler

## Experiment Parameters

We fix all parameters up front so every section of this notebook uses a consistent setup. We define :

- `CONFIDENCE_LEVEL` : the confidence level at which we will compute confidence intervals.

- `N_TOTAL` : the total number of samples in the pool. Every sample has a proxy prediction and an oracle proxy uncertainty.

- `N_LABELED` : the number of labeled samples in the pool.

- `TRUE_MEAN` : the true mean value of human labels.

- `PROXY_MEAN` : the (biased) proxy mean value.

- `N_SEEDS` : the number of simulations we will run in our Monte Carlo experiments.

> **Note on correlation bounds:** Depending on the values of `TRUE_MEAN` and `PROXY_MEAN`, extreme correlation values (close to -1 or 1) may not be achievable. Correlation sweeps are kept within these limits.

Finally, we define the baseline estimation methods for comparison:

- `True only` : uses `N_LABELED` actively sampled true labels with an IPW-corrected classical CLT confidence interval. No proxy labels are used.

- `Proxy only` : uses proxy labels only, without correction.

- `ASI` : Active Statistical Inference, the same actively sampled true labels as `True only`, further combined with IPW-rectified proxy labels for additional efficiency.

In [ ]:
CONFIDENCE_LEVEL = 0.9
N_TOTAL = 4200  # total pool size (all samples have oracle uncertainty)
N_LABELED = 200  # labeling budget
TRUE_MEAN = 0.55
PROXY_MEAN = 0.5
N_SEEDS = 1000

METHODS = ["True only", "Proxy only", "ASI"]

correlations = np.arange(0.1, 0.95, 0.1)
n_correlations = len(correlations)
correlations_lmh = [
    correlations[n_correlations // 4],
    correlations[n_correlations // 2],
    correlations[3 * n_correlations // 4],
]  # low, medium and high values
corr_labels = ["Low", "Medium", "High"]

## Data Generation

We use `generate_binary_dataset_with_oracle_sampling` to simulate a realistic evaluation scenario.
It returns three parallel arrays of length `N_TOTAL`, one value per sample:
- `y_true_oracle` ($Y$) : ground-truth binary label (latent, revealed only for labeled samples)
- `y_proxy` ($\tilde{Y}$) : proxy binary prediction (always available for every sample)
- `uncertainty` : oracle proxy uncertainty $\sqrt{\mathbb{E}[(\tilde{Y}_i - Y_i)^2 \mid x_i]}$, quantifies per-sample proxy reliability

Samples with high `uncertainty` are those where the proxy is least reliable. The `True only` and `ASI` methods assign higher labeling probability to these samples via `ActiveSampler`:

$$\pi_i = \min\Big(1, \frac{N_{\text{labeled}} \cdot \text{uncertainty}_i}{\sum_j \text{uncertainty}_j}\Big), \quad \pi_i \in (0, 1]$$

This concentrates the labeling budget on samples where true labels add the most information.

The `build_dataset` helper below applies `ActiveSampler` to the `uncertainty` to compute sampling probabilities $\pi_i$ and Bernoulli selection indicators $\xi_i$ for each sample. Samples with $\xi_i = 0$ have their `y_true_oracle` value set to `np.nan` (unobserved).

In [ ]:
def build_dataset(y_true_oracle, y_proxy, uncertainty, seed):

    # Active sampling via ActiveSampler — shared by True only and ASI
    pi, xi = ActiveSampler().sample(uncertainty, budget=N_LABELED, random_seed=seed)

    y_true = y_true_oracle.copy()
    y_true[~(xi.astype(bool))] = np.nan

    return y_true, y_proxy, pi

We now use the previous function to generate a single example dataset for illustration with correlation = 0.5

In [ ]:
y_true_oracle, y_proxy, uncertainty = generate_binary_dataset_with_oracle_sampling(
    N=N_TOTAL,
    true_mean=TRUE_MEAN,
    proxy_mean=PROXY_MEAN,
    correlation=0.5,
    random_seed=0,
)

y_true, y_proxy, pi_example = build_dataset(y_true_oracle, y_proxy, uncertainty, seed=0)

n_labeled = int(np.sum(~np.isnan(y_true)))

Now print some statistics about the labeling budget and sampling probabilities

In [ ]:
print(f"Total samples      : {N_TOTAL}")
print(f"Labeling budget  : {N_LABELED}")
print(f"Labeled (realized, Bernoulli) : {n_labeled}")
print(
    f"\nSampling probability p, — min: {pi_example.min():.3f},"
    f" max: {pi_example.max():.3f}, mean: {pi_example.mean():.3f}"
)

Let's look at how the active sampling probability $\pi_i$ is distributed across samples in this example. Since both `True only` and `ASI` share this sampling rule, this distribution applies to both.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pi_example, bins=30, color="darkorange", alpha=0.75, label="Active sampling probability $\\pi_i$")
ax.set_xlabel("Sampling probability $\\pi_i$")
ax.set_ylabel("Count")
ax.set_title("Distribution of active sampling probabilities  (correlation = 0.5)")
ax.legend()
plt.tight_layout()
plt.show()

The histogram shows that $\pi_i$ values are spread around the mean labeling rate. Samples where the proxy is unreliable (high `uncertainty`) receive a higher sampling probability, while samples where the proxy is already reliable receive a lower one.

In the following sections, we will perform Monte Carlo experiments to estimate confidence interval width among other things.

This consists in running `N_SEEDS` simulations where we generate data, compute a confidence interval and measure its width each time. We end up with `N_SEEDS` sample values for the measured quantity that we can use to compute statistics.

The same method can be used to evaluate coverage which will be defined and illustrated below.

## Inference Results

All three methods receive the same labeled samples (drawn with the same active sampling rule). Their differences are summarised below:

| Estimation method | Data used | Notes |
|--------|-----------|-------|
| **True only** | `y_true` (active sampling, IPW-corrected) | No proxy labels |
| **Proxy only** | `y_proxy` | Biased, cheap but wrong |
| **ASI** | `y_true` (active sampling, IPW-rectified) + `y_proxy` | Same labels as True only, plus proxy rectification |

For a given set of arrays, the following function computes the outputs of each method.

In [ ]:
def generate_estimates(y_true, y_proxy, sampling_probability):
    """Return mean and std for all three estimation methods."""
    # --- ASI (active sampling, IPW-corrected proxy rectification) ---
    asi_result = ASIMeanEstimator().estimate(
        y_true,
        y_proxy,
        sampling_probability,
        confidence_level=CONFIDENCE_LEVEL,
    )

    # --- True only (active sampling, IPW-corrected, no proxy) ---
    true_only_result = IPWClassicalMeanEstimator().estimate(
        y_true, sampling_probability, confidence_level=CONFIDENCE_LEVEL
    )

    # --- Proxy only (no sampling correction, biased) ---
    proxy_only_result = ClassicalMeanEstimator().estimate(y_proxy, confidence_level=CONFIDENCE_LEVEL)

    return {
        "True only": {
            "mean": true_only_result.mean,
            "std": true_only_result.std,
        },
        "Proxy only": {
            "mean": proxy_only_result.mean,
            "std": proxy_only_result.std,
        },
        "ASI": {
            "mean": asi_result.mean,
            "std": asi_result.std,
            "effective_sample_size": asi_result.effective_sample_size,
        },
    }

ASI is implemented by the `ASIMeanEstimator` whereas `IPWClassicalMeanEstimator` implements IPW-corrected mean estimation and `ClassicalMeanEstimator` implements conventional mean estimation.

The three next functions implement the process of verification of valid confidence intervals for each estimation method step by step.

- `monte_carlo_simulation` simulates data for a single correlation level, applies each method and returns their outputs for each simulation.

- `compute_hits` takes the output of `monte_carlo_simulation` as input in addition to a confidence level and computes the hits on the associated confidence intervals i.e. the times where the target estimated value (`TRUE_MEAN`) is within the interval and the times where it isn't.

- `coverage_with_errbar` takes the output of `compute_hits` as input and estimates the mean coverage values i.e. the proportion of times confidence intervals contained the target value. This estimation also provides a confidence interval for the mean coverage.

In [ ]:
def monte_carlo_simulation(correlation: float, n_seeds=N_SEEDS):
    """Single Monte Carlo loop: cache per-seed mean, std, and effective sample size for each estimation method."""
    means = {method: np.zeros(n_seeds) for method in METHODS}
    stds = {method: np.zeros(n_seeds) for method in METHODS}
    effective_sample_size = np.zeros(n_seeds)

    for seed in range(n_seeds):
        y_true, y_proxy, uncertainty = generate_binary_dataset_with_oracle_sampling(
            N=N_TOTAL,
            true_mean=TRUE_MEAN,
            proxy_mean=PROXY_MEAN,
            correlation=correlation,
            random_seed=seed,
        )
        y_true, y_proxy, sampling_probability = build_dataset(y_true, y_proxy, uncertainty, seed)
        estimates = generate_estimates(y_true, y_proxy, sampling_probability)
        for method in METHODS:
            means[method][seed] = estimates[method]["mean"]
            stds[method][seed] = estimates[method]["std"]
        effective_sample_size[seed] = estimates["ASI"]["effective_sample_size"]

    stats = {method: {"means": means[method], "stds": stds[method]} for method in METHODS}
    stats["ASI"]["effective_sample_size"] = effective_sample_size
    return stats


def compute_hits(stats, confidence_level):
    """Return per-seed hit indicators {method: float array} at the given confidence level."""
    hits = {}
    for method in METHODS:
        ci = CLTConfidenceInterval(
            mean=stats[method]["means"], std=stats[method]["stds"], confidence_level=confidence_level
        )
        hits[method] = (ci.lower_bound <= TRUE_MEAN) & (TRUE_MEAN <= ci.upper_bound)
    return hits


def coverage_with_errbar(hits, confidence_level):
    """Estimate empirical coverage + Confidence Interval via ClassicalMeanEstimator
    on per-seed hit indicators."""
    estimator = ClassicalMeanEstimator()
    r = estimator.estimate(hits, confidence_level=confidence_level)
    return r.mean, r.confidence_interval.lower_bound, r.confidence_interval.upper_bound

## Coverage Validity

A confidence interval is **valid** if it is built using an estimation method that reliably captures the true value. For example, a 90% confidence interval is valid if, when you repeat the experiment many times and compute a new interval each time, around 90% of those intervals end up containing the true value.

We check this empirically via a Monte Carlo experiment as described above and count how often each method's confidence interval covers `TRUE_MEAN`.

> **Key question:** Does ASI maintain valid coverage under non-uniform oracle sampling, or does the biased selection inflate or deflate coverage?

The IPW correction is such that coverage is maintained. The sampling probabilities are used to de-bias the oracle-selected estimates restoring validity as in uniform sampling.

### Coverage vs confidence level for three correlation levels

We sweep the confidence level from 0.55 to 0.95 and plot the observed coverage.
For a valid estimation method, the dots should fall on or around the black diagonal $y = \text{confidence level}$.

We do this for **low**, **medium** and **high** proxy correlation.

In [ ]:
# Run Monte Carlo simulations for each correlation level
raw_stats = {correlation: monte_carlo_simulation(correlation) for correlation in correlations}

confidence_levels = np.arange(0.55, 1.00, 0.05)

# Derive coverage for every (correlation, confidence_level) pair
coverages_confidence_intervals = {}
for correlation in correlations_lmh:
    coverages_confidence_intervals[correlation] = {}
    for confidence_level in confidence_levels:
        hits = compute_hits(raw_stats[correlation], confidence_level)
        coverages_confidence_intervals[correlation][confidence_level] = {}
        for method in METHODS:
            coverage_confidence_interval = coverage_with_errbar(hits[method], confidence_level=confidence_level)
            coverages_confidence_intervals[correlation][confidence_level][method] = coverage_confidence_interval

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
colors = {"True only": "steelblue", "ASI": "darkorange", "Proxy only": "red"}

for ax, correlation, label in zip(axes, correlations_lmh, corr_labels):
    ax.plot(confidence_levels, confidence_levels, color="black", lw=1.5, linestyle="--", label="Ideal")
    for method in METHODS:
        mean = [coverages_confidence_intervals[correlation][cl][method][0] for cl in confidence_levels]
        lo = [coverages_confidence_intervals[correlation][cl][method][1] for cl in confidence_levels]
        hi = [coverages_confidence_intervals[correlation][cl][method][2] for cl in confidence_levels]

        ax.plot(confidence_levels, mean, marker="o", color=colors[method], label=method)
        ax.fill_between(confidence_levels, lo, hi, alpha=0.15, color=colors[method])

    ax.set_xlabel("Target confidence level")
    ax.set_ylabel("Observed coverage")
    ax.set_title(f"{label} correlation (${round(correlation, 2)}$)")
    ax.legend(fontsize=8)
    ax.set_xlim(0.5, 1.0)
    ax.set_ylim(0.5, 1.0)

fig.suptitle("Empirical coverage vs target confidence level", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

Both **ASI** and **True only** track the diagonal closely across all correlation levels, confirming that ASI achieves valid coverage regardless of proxy quality. The Proxy only method does not show up because it uses biased data so that its coverage is invalid (close to zero).

Since both **ASI** and **True only** use the same active sampling rule, this comparison directly isolates the effect of incorporating proxy labels: ASI's validity is preserved even after adding the proxy rectification step.

---

### Coverage vs correlation for fixed confidence level

We now fix the confidence level at 90% and vary the proxy-true correlation from 0.1 to 0.9.
This shows that ASI's validity does not degrade as the proxy becomes weaker.

In [ ]:
coverage_by_corr = {}  # {correlation: {method: observed mean coverage}}
coverage_ci_by_corr = {}  # {correlation: {method: (lower, upper) Confidence Interval on coverage}}

for correlation in correlations:
    hits = compute_hits(raw_stats[correlation], CONFIDENCE_LEVEL)
    coverage_by_corr[correlation] = {}
    coverage_ci_by_corr[correlation] = {}
    for method in METHODS:
        mean_cov, lo, hi = coverage_with_errbar(hits[method], CONFIDENCE_LEVEL)
        coverage_by_corr[correlation][method] = mean_cov
        coverage_ci_by_corr[correlation][method] = (lo, hi)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
method_colors = {"True only": "steelblue", "Proxy only": "red", "ASI": "darkorange"}

for method in ["True only", "ASI"]:
    obs = [coverage_by_corr[correlation][method] for correlation in correlations]
    lo = [coverage_ci_by_corr[correlation][method][0] for correlation in correlations]
    hi = [coverage_ci_by_corr[correlation][method][1] for correlation in correlations]
    ax.plot(correlations, obs, marker="o", color=method_colors[method], label=method)
    ax.fill_between(correlations, lo, hi, alpha=0.15, color=method_colors[method])

ax.axhline(y=CONFIDENCE_LEVEL, color="red", linestyle="--", lw=2, label=f"Target coverage {CONFIDENCE_LEVEL:.0%}")
ax.set_xlabel("Proxy–true correlation")
ax.set_ylabel("Observed coverage")
ax.set_title(f"Coverage vs correlation  (confidence level = {CONFIDENCE_LEVEL:.0%})")
ax.set_xlim(0, 1)
ax.set_ylim(0.8, 1.0)
ax.legend()
plt.tight_layout()
plt.show()

Note that **Proxy only** is not plotted because the proxy is biased (proxy mean ≠ true mean). Therefore it has invalid coverage (close to 0) whereas **ASI** and **True only** remain valid across all correlation levels.

---

## Confidence Interval Width

Coverage validity is necessary but not sufficient, we also want **short** intervals. Both **True only** and **ASI** use the same active sampling, so any width difference between them is attributable solely to the proxy labels: ASI uses the proxy signal to rectify the estimate, extracting additional information beyond what the true labels alone provide.

We report the **mean** and a **percentile band** to capture variability.

In [ ]:
width_by_corr = {}
for correlation in correlations:
    width_by_corr[correlation] = {}
    for method in METHODS:
        ci = CLTConfidenceInterval(
            mean=raw_stats[correlation][method]["means"],
            std=raw_stats[correlation][method]["stds"],
            confidence_level=CONFIDENCE_LEVEL,
        )
        width_by_corr[correlation][method] = ci.upper_bound - ci.lower_bound

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_methods_w = ["True only", "ASI"]
colors_w = {"True only": "steelblue", "ASI": "darkorange"}

# Compute percentiles based on CONFIDENCE_LEVEL\n",
lower_percentile = round(((1 - CONFIDENCE_LEVEL) / 2) * 100)
upper_percentile = 100 - lower_percentile

for method in plot_methods_w:
    means_w = [np.mean(width_by_corr[correlation][method]) for correlation in correlations]
    q_lower = [np.percentile(width_by_corr[correlation][method], lower_percentile) for correlation in correlations]
    q_upper = [np.percentile(width_by_corr[correlation][method], upper_percentile) for correlation in correlations]
    ax.plot(correlations, means_w, marker="o", label=method, color=colors_w[method])
    ax.fill_between(correlations, q_lower, q_upper, alpha=0.15, color=colors_w[method])

ax.set_xlabel("Proxy–true correlation")
ax.set_ylabel("Confidence Interval width")
ax.set_title(
    f"Confidence interval width vs correlation  (confidence level = {CONFIDENCE_LEVEL:.0%})\n"
    f"Solid = mean, shaded = {lower_percentile}th–{upper_percentile}th percentile"
)
ax.set_xlim(0.05, 0.95)
ax.legend()
plt.tight_layout()
plt.show()

As expected, ASI's interval width decreases with increasing correlation. Since both methods share the same active sample, the width reduction is entirely due to the proxy rectification step. Note that benefits can be seen mainly for reasonable correlation values (> 0.4).

---

## Effective Sample Size

A natural summary of ASI's efficiency gain is the **Effective Sample Size (ESS)**: *it is the number of true labels needed to achieve the same confidence interval width as ASI with the current budget.*

Since confidence interval width $\propto 1/\sqrt{n}$, we can estimate ESS empirically as:

$$\text{ESS} = N_{\text{labeled}} \times \left(\frac{\bar{w}_{\text{True only, unif}}}{\bar{w}_{\text{ASI}}}\right)^2,$$

where $\bar{w}_{\text{ASI}}$ and $\bar{w}_{\text{True only, unif}}$ are the confidence interval widths for ASI and using uniformly sampled true labels respectively.

The ESS purely reflects the information added by the proxy labels. When the correlation is zero, ESS $\approx N_{\text{labeled}}$ (proxy adds nothing). As the correlation approaches $1,~$ ESS grows. ASI can be equivalent to having a much larger labeled dataset.

In [ ]:
ess_mean = [np.mean(raw_stats[correlation]["ASI"]["effective_sample_size"]) for correlation in correlations]
ess_q_lower = [
    np.percentile(raw_stats[correlation]["ASI"]["effective_sample_size"], lower_percentile)
    for correlation in correlations
]
ess_q_upper = [
    np.percentile(raw_stats[correlation]["ASI"]["effective_sample_size"], upper_percentile)
    for correlation in correlations
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(correlations, ess_mean, marker="o", color="darkorange", label="ASI ESS (mean)")
ax.fill_between(
    correlations,
    ess_q_lower,
    ess_q_upper,
    alpha=0.15,
    color="darkorange",
    label=f"{lower_percentile}th–{upper_percentile}th percentile",
)
ax.axhline(y=N_LABELED, color="steelblue", linestyle="--", lw=2, label=f"Baseline (True only, n={N_LABELED})")
ax.set_xlabel("Proxy–true correlation")
ax.set_ylabel("Effective sample size")
ax.set_title("ASI effective sample size vs proxy correlation")
ax.set_xlim(0.05, 0.95)
ax.legend()
plt.tight_layout()
plt.show()

## Summary

This notebook has empirically validated that GLIDE's ASI implementation satisfies two key statistical properties:

| Property | Result |
|----------|--------|
| **Coverage validity** | ASI achieves the nominal coverage across all correlation levels and confidence levels tested |
| **Efficiency** | ASI produces shorter confidence intervals than `True only` for sufficient correlation levels, with the gain growing with correlation |

Because both **ASI** and **True only** share the same active sampling rule, every observed difference (in interval width or effective sample size) is attributable exclusively to the proxy labels. This clean comparison confirms that the proxy rectification step in ASI adds genuine statistical efficiency without sacrificing validity.

Crucially, the biased baseline (**Proxy only**) fails the coverage test. It appears precise but is systematically wrong. ASI avoids this by correcting for proxy bias via IPW using the labeled subset.

The ESS analysis shows that with a proxy correlation of $0.9,$ ASI is equivalent to having more than **twice more** labeled data, a practical gain in scenarios where true annotation is expensive. This highlights the importance of a good LLM judge to evaluate an AI system.